# London Airbnb Price Prediction (Regression)

**Goal:** predict the nightly price of a London Airbnb listing from its features.

**Data:** Inside Airbnb, London `listings.csv.gz` (17 September 2026). Contains Inside Airbnb data, licensed under Creative Commons. Source: insideairbnb.com

**Models compared:** Linear Regression, Ridge, Random Forest, Gradient Boosting
**Metrics:** R², MAE, RMSE

Keep `listings.csv.gz` in the same folder as this notebook.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## 2. Load the data

In [ ]:
df = pd.read_csv("listings.csv.gz", compression="gzip", low_memory=False)
print(df.shape)
df[["name","room_type","neighbourhood_cleansed","accommodates","bedrooms","price"]].head()

## 3. Data cleaning

Findings from inspecting the raw file:
- `price` is text such as `$267.50`, and about 35% of listings have no price, so they cannot be used for training.
- Several columns are completely empty (e.g. `bathrooms`, `host_since`, `instant_bookable`), so bathrooms are parsed from `bathrooms_text` instead.
- A few extreme prices exist (up to 75,000 per night), so listings are kept between 20 and 1,000 per night.

In [ ]:
# Convert price text to a number
df["price"] = df["price"].str.replace(r"[\$,]", "", regex=True).astype(float)

print("Missing price:", round(df["price"].isna().mean() * 100, 1), "%")
print(df["price"].describe(percentiles=[.01, .5, .95, .99]))

In [ ]:
df = df.dropna(subset=["price"]).drop_duplicates(subset="id")
df = df[(df["price"] >= 20) & (df["price"] <= 1000)].copy()
print("Rows after cleaning:", len(df))

## 4. Feature engineering

In [ ]:
# Bathrooms from text like "1.5 shared baths"
df["bathrooms_num"] = df["bathrooms_text"].str.extract(r"(\d+\.?\d*)")[0].astype(float)
df.loc[df["bathrooms_text"].str.contains("half", case=False, na=False), "bathrooms_num"] = 0.5
df["shared_bath"] = df["bathrooms_text"].str.contains("shared", case=False, na=False).astype(int)

# Number of amenities
df["amenities_count"] = df["amenities"].str.count(",") + 1

# Superhost flag
df["superhost"] = (df["host_is_superhost"] == "t").astype(int)

# Approximate distance to central London (km)
centre_lat, centre_lon = 51.5074, -0.1278
df["dist_centre_km"] = np.sqrt(((df["latitude"] - centre_lat) * 111) ** 2 +
                               ((df["longitude"] - centre_lon) * 69.4) ** 2)

# Group rare property types
top_types = df["property_type"].value_counts().head(12).index
df["property_group"] = np.where(df["property_type"].isin(top_types), df["property_type"], "Other")

## 5. Exploratory analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["price"], bins=50, ax=ax[0])
ax[0].set_title("Nightly price")
sns.histplot(np.log1p(df["price"]), bins=50, ax=ax[1])
ax[1].set_title("Log of nightly price (more symmetric)")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="room_type", y="price")
plt.title("Price by room type"); plt.xticks(rotation=15); plt.show()

In [ ]:
top_n = df.groupby("neighbourhood_cleansed")["price"].median().sort_values(ascending=False).head(10)
plt.figure(figsize=(8, 4))
sns.barplot(x=top_n.values, y=top_n.index, color="steelblue")
plt.title("10 most expensive boroughs (median nightly price)")
plt.xlabel("Median price"); plt.show()

In [ ]:
num_cols = ["price", "accommodates", "bedrooms", "beds", "bathrooms_num", "amenities_count",
            "number_of_reviews", "review_scores_rating", "dist_centre_km", "availability_365"]
plt.figure(figsize=(9, 7))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation heatmap"); plt.show()

## 6. Train / test split

In [ ]:
num = ["accommodates", "bedrooms", "beds", "bathrooms_num", "shared_bath", "amenities_count",
       "minimum_nights", "availability_365", "number_of_reviews", "review_scores_rating",
       "review_scores_location", "reviews_per_month", "superhost",
       "calculated_host_listings_count", "hosts_time_as_host_years",
       "dist_centre_km", "latitude", "longitude"]
cat = ["room_type", "property_group", "neighbourhood_cleansed"]

X = df[num + cat]
y = np.log1p(df["price"])      # model log price, convert back later

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

Note: `estimated_revenue_l365d` is deliberately **not** used as a feature, because it is calculated from price and would leak the answer.

## 7. Build and compare models

In [ ]:
pre_linear = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])

pre_tree = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])

models = {
    "Linear Regression": Pipeline([("pre", pre_linear), ("model", LinearRegression())]),
    "Ridge": Pipeline([("pre", pre_linear), ("model", Ridge(alpha=10))]),
    "Random Forest": Pipeline([("pre", pre_tree), ("model", RandomForestRegressor(
        n_estimators=150, min_samples_leaf=2, n_jobs=-1, random_state=42))]),
    "Gradient Boosting": Pipeline([("pre", pre_tree), ("model", HistGradientBoostingRegressor(
        max_iter=400, learning_rate=0.08, random_state=42))]),
}

rows, preds = [], {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    p = pipe.predict(X_test)
    actual, pred_price = np.expm1(y_test), np.expm1(p)
    preds[name] = pred_price
    rows.append({"Model": name,
                 "R2 (log price)": r2_score(y_test, p),
                 "R2 (price)": r2_score(actual, pred_price),
                 "MAE": mean_absolute_error(actual, pred_price),
                 "RMSE": np.sqrt(mean_squared_error(actual, pred_price))})

results = pd.DataFrame(rows).round(3).sort_values("R2 (price)", ascending=False)
results

## 8. Check for overfitting

In [ ]:
best_name = results.iloc[0]["Model"]
best = models[best_name]
print("Best model:", best_name)
print("Train R2 (log price):", round(r2_score(y_train, best.predict(X_train)), 3))
print("Test  R2 (log price):", round(r2_score(y_test, best.predict(X_test)), 3))

## 9. Actual vs predicted

In [ ]:
actual = np.expm1(y_test)
plt.figure(figsize=(6, 6))
plt.scatter(actual, preds[best_name], alpha=0.15, s=8)
plt.plot([0, 1000], [0, 1000], "r--")
plt.xlabel("Actual price"); plt.ylabel("Predicted price")
plt.title(f"Actual vs predicted ({best_name})"); plt.show()

## 10. Feature importance

In [ ]:
from sklearn.inspection import permutation_importance

sample = X_test.sample(3000, random_state=1)
imp = permutation_importance(best, sample, y_test.loc[sample.index],
                             n_repeats=5, random_state=42, n_jobs=-1)
importance = pd.Series(imp.importances_mean, index=sample.columns).sort_values(ascending=False).head(12)

plt.figure(figsize=(8, 5))
sns.barplot(x=importance.values, y=importance.index, color="darkorange")
plt.title("What drives London Airbnb prices? (permutation importance)")
plt.xlabel("Drop in R2 when the feature is shuffled")
plt.tight_layout(); plt.show()

## 11. Conclusions and limitations

- Gradient Boosting and Random Forest clearly beat the linear models, which suggests price depends on non-linear effects and interactions (for example, location combined with size).
- Room type, distance to central London, and size (`accommodates`, `bedrooms`) are the strongest price drivers.
- **Limitations:** about 35% of listings had no price and were dropped; listings above 1,000 per night were removed; the data is a single snapshot, so seasonality is not captured; there is no text or photo information.
- **Next steps:** tune hyperparameters, add features from listing descriptions, and try a time-aware model using the calendar file.